# HC-3 Download And Phase Target Prototype

This notebook starts the GRTPL workflow:

1. Download public CRCNS HC-3 documentation and metadata.
2. Inspect channel-order metadata before choosing adjacent channel pairs.
3. Prepare a selected session download path without pulling the full dataset.
4. Smoke-test the causal/acausal phase target logic on a synthetic theta signal.

Full HC-3 LFP data is large. Keep raw data under `data/raw/`, which is ignored by git.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

project_root

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zipfile import ZipFile

from grtpl.hc3 import download_public_hc3_docs, hc3_session_url
from grtpl.signal import causal_bandpass, acausal_bandpass, hilbert_phase, decimate_by_timestamp
from grtpl.targets import find_next_phase_crossings, next_crossing_for_samples, nominal_phase_targets

## Download Public HC-3 Docs

These files are small enough to fetch immediately and include the data description, metadata tables, channel order, and probe geometry.

In [ ]:
raw_dir = project_root / "data" / "raw"
downloads = download_public_hc3_docs(raw_dir)
pd.DataFrame([d.__dict__ for d in downloads])

## Inspect Channel Metadata

The next practical decision is selecting adjacent hippocampal channels. Start by listing the channel-order files.

In [ ]:
channel_zip = raw_dir / "hc3" / "docs" / "crcns-hc3-channelorder.zip"
with ZipFile(channel_zip) as zf:
    channel_files = zf.namelist()

channel_files[:20], len(channel_files)

## Inspect Session Metadata

The HC-3 metadata tables include a file/session table. The CSV files do not all include headers, so this first pass keeps them raw until we map columns from the data description.

In [ ]:
metadata_zip = raw_dir / "hc3" / "docs" / "crcns-hc3-metadata-tables.zip"
with ZipFile(metadata_zip) as zf:
    metadata_files = [name for name in zf.namelist() if name.endswith((".csv", ".txt"))]

metadata_files

In [ ]:
with ZipFile(metadata_zip) as zf:
    with zf.open("crcns-hc3-metadata-tables/hc3-file.csv") as handle:
        hc3_file_table = pd.read_csv(handle, header=None)

hc3_file_table.head()

## Prepare Selected Session Download

Once we choose a small first HC-3 session archive, set `session_archive_name` below. The NERSC mirror pattern is generated here, but the actual download is left explicit because the full data is large and may require CRCNS credentials.

In [ ]:
session_archive_name = "TODO-select-small-session-archive.zip"
hc3_session_url(session_archive_name)

## Synthetic Phase Target Smoke Test

This validates the mechanics before we bind to HC-3 file parsing. It does not claim to be the final causal phase estimator; it is a correctness scaffold for the target construction.

In [ ]:
sample_rate_hz = 1250.0
duration_s = 20.0
theta_hz = 8.0
rng = np.random.default_rng(7)

timestamps = np.arange(0, duration_s, 1 / sample_rate_hz)
lfp = np.sin(2 * np.pi * theta_hz * timestamps) + 0.15 * rng.standard_normal(len(timestamps))

causal_theta, sos, zf = causal_bandpass(lfp, sample_rate_hz, band_hz=(6.0, 10.0), order=4)
reference_theta = acausal_bandpass(lfp, sample_rate_hz, band_hz=(6.0, 10.0), order=4)

causal_phase = hilbert_phase(causal_theta)
reference_phase = hilbert_phase(reference_theta)

input_rate_hz = 25.0
input_timestamps, input_phase = decimate_by_timestamp(timestamps, causal_phase, input_rate_hz)

crossing_timestamps, crossing_indices = find_next_phase_crossings(timestamps, reference_phase, target_phase_rad=np.pi)
target_timestamps = next_crossing_for_samples(input_timestamps, crossing_timestamps, max_horizon_s=0.5)
target_phase = nominal_phase_targets(input_phase, input_timestamps, target_timestamps, nominal_frequency_hz=theta_hz)

pd.DataFrame({
    "input_time_s": input_timestamps,
    "causal_phase_deg": np.rad2deg(input_phase) % 360,
    "target_time_s": target_timestamps,
    "target_nominal_phase_deg": np.rad2deg(target_phase) % 360,
}).head(12)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
window = timestamps < 2.0
axes[0].plot(timestamps[window], lfp[window], color="0.65", label="raw synthetic LFP")
axes[0].plot(timestamps[window], causal_theta[window], label="causal 6-10 Hz")
axes[0].plot(timestamps[window], reference_theta[window], label="acausal reference", alpha=0.8)
axes[0].legend(loc="upper right")
axes[0].set_ylabel("signal")

phase_window = input_timestamps < 2.0
axes[1].step(input_timestamps[phase_window], np.rad2deg(input_phase[phase_window]) % 360, where="post")
axes[1].set_ylabel("causal phase deg")
axes[1].set_xlabel("time s")

fig.tight_layout()